# AI-Powered Python-to-C++ Code Generator

An LLM-powered application that translates Python code into C++ and provides a workflow for compiling and benchmarking the generated implementation.

The notebook supports multiple model providers through OpenAI-compatible chat completion APIs. Generated C++ is compiled locally with `clang++`.


In [ ]:
import io
import os
import platform
import re
import subprocess
import sys

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI


In [ ]:
load_dotenv()

api_keys = {
    "openai": os.getenv("OPENAI_API_KEY"),
    "anthropic": os.getenv("ANTHROPIC_API_KEY"),
    "google": os.getenv("GOOGLE_API_KEY"),
    "grok": os.getenv("GROK_API_KEY"),
    "groq": os.getenv("GROQ_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY"),
}

missing_keys = [provider for provider, key in api_keys.items() if not key]
if missing_keys:
    print(f"API keys not configured (optional providers): {', '.join(missing_keys)}")


In [ ]:
clients = {
    "openai": OpenAI(),
    "anthropic": OpenAI(
        api_key=api_keys["anthropic"],
        base_url="https://api.anthropic.com/v1/",
    ),
    "gemini": OpenAI(
        api_key=api_keys["google"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    ),
    "grok": OpenAI(
        api_key=api_keys["grok"],
        base_url="https://api.x.ai/v1",
    ),
    "groq": OpenAI(
        api_key=api_keys["groq"],
        base_url="https://api.groq.com/openai/v1",
    ),
    "ollama": OpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1",
    ),
    "openrouter": OpenAI(
        api_key=api_keys["openrouter"],
        base_url="https://openrouter.ai/api/v1",
    ),
}


In [ ]:
models = [
    "gpt-6.1-sol",
    "claude-sonnet-5-5",
    "grok-4.3",
    "gemini-4-argon",
    "qwen2.5-coder",
    "deepseek-coder-v2",
    "gpt-oss:20b",
    "qwen/qwen3-coder-30b-a3b-instruct",
    "openai/gpt-oss-120b",
]

model_clients = {
    "gpt-6.1-sol": clients["openai"],
    "claude-sonnet-5-5": clients["anthropic"],
    "grok-4.3": clients["grok"],
    "gemini-4-argon": clients["gemini"],
    "qwen2.5-coder": clients["ollama"],
    "deepseek-coder-v2": clients["ollama"],
    "gpt-oss:20b": clients["ollama"],
    "qwen/qwen3-coder-30b-a3b-instruct": clients["openrouter"],
    "openai/gpt-oss-120b": clients["groq"],
}


In [ ]:
system_info = {
    "operating_system": platform.platform(),
    "processor": platform.processor() or platform.machine(),
    "architecture": platform.machine(),
    "python_version": platform.python_version(),
}
system_info


In [ ]:
compile_command = [
    "clang++",
    "-std=c++17",
    "-O3",
    "-mcpu=native",
    "main.cpp",
    "-o",
    "main",
]
run_command = ["./main"]


In [ ]:
system_prompt = """
You convert Python code into high-performance C++ code.
The generated C++ program must preserve the Python program's observable output and behaviour.
Return only valid C++ source code, with brief comments only where useful.
"""


def user_prompt_for(python_code):
    return f"""
Translate the following Python code into an efficient C++17 implementation.

Target system:
{system_info}

Compilation command:
{compile_command}

Return only C++ source code.

Python code:
```python
{python_code}
```
"""


In [ ]:
def messages_for(python_code):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python_code)},
    ]


In [ ]:
def write_output(cpp_code, output_path="main.cpp"):
    with open(output_path, "w", encoding="utf-8") as output_file:
        output_file.write(cpp_code)


In [ ]:
def port(model, python_code):
    client = model_clients[model]

    request_options = {
        "model": model,
        "messages": messages_for(python_code),
    }
    if model == "gpt-6.1-sol":
        request_options["reasoning_effort"] = "high"

    response = client.chat.completions.create(**request_options)
    cpp_code = response.choices[0].message.content or ""
    cpp_code = re.sub(r"^```(?:cpp|c\+\+)?\s*|\s*```$", "", cpp_code.strip(), flags=re.IGNORECASE)

    write_output(cpp_code)
    return cpp_code


In [ ]:
python_example = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= 1 / j
        j = i * param1 + param2
        result += 1 / j
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {end_time - start_time:.6f} seconds")
"""


In [ ]:
def compile_and_run():
    try:
        subprocess.run(
            compile_command,
            check=True,
            text=True,
            capture_output=True,
        )
        for _ in range(3):
            result = subprocess.run(
                run_command,
                check=True,
                text=True,
                capture_output=True,
            )
            print(result.stdout)
    except subprocess.CalledProcessError as error:
        print(f"Command failed: {error.cmd}")
        print(error.stderr)


In [ ]:
with gr.Blocks(title="Python-to-C++ Code Generator") as ui:
    with gr.Row():
        python_input = gr.Textbox(
            label="Python code",
            lines=28,
            value=python_example,
        )
        cpp_output = gr.Code(
            label="Generated C++ code",
            language="cpp",
            lines=28,
        )

    with gr.Row():
        model_selector = gr.Dropdown(
            choices=models,
            label="Select model",
            value=models[0],
        )
        convert_button = gr.Button("Convert code", variant="primary")

    convert_button.click(
        port,
        inputs=[model_selector, python_input],
        outputs=cpp_output,
    )

ui.launch(inbrowser=True)


In [ ]:
compile_and_run()

## Reference Benchmark Results

The following execution times were recorded in the original experiment. Results depend on the hardware, compiler, model version, and runtime environment.

| Model | Execution time (seconds) |
|---|---:|
| Qwen 2.5 Coder | Failed |
| DeepSeek Coder v2 | 0.114050084 |
| OpenAI GPT-OSS 20B | 0.080438 |
| Qwen3 Coder 30B | 0.113734 |
| OpenAI GPT-OSS 120B | 1.407383 |


## Reference Model Performance Comparison

The speedups below are rankings reported in the original experiment. They are reference figures, not benchmark results measured by this notebook on the current machine.

| Rank | Model | Reported speedup |
|---:|---|---:|
| 9 | Qwen 2.5 Coder | Failed |
| 8 | OpenAI GPT-OSS 120B | 14x |
| 7 | DeepSeek Coder v2 | 168x |
| 6 | Qwen3 Coder 30B | 168x |
| 5 | Claude Sonnet 4.5 | 184x |
| 4 | GPT-5 | 233x |
| 3 | GPT-OSS 20B | 238x |
| 2 | Grok 4 | 1,060x |
| 1 | Gemini 2.5 Pro | 1,440x |
